# Jobbannons-analys för AI-utvecklare

**Kursmoment:** Utveckling med Python, grund

**Syfte:** Hämta jobbannonser från JobTech API (Arbetsförmedlingen), analysera vilka teknologier som efterfrågas för AI-relaterade roller, spara resultatet som CSV och visualisera det med ett diagram.

**Så här är notebooken upplagd:**
1. Import av bibliotek
2. Utforska API:et (testar och tittar på datan innan vi bygger vidare)
3. Klasser – ett enkelt sätt att hålla ihop informationen om varje jobbannons
4. Funktioner – varje funktion gör EN sak (hämta, analysera, spara, rita diagram)
5. Huvudprogram – kör alla funktioner i rätt ordning

Koden är skriven med enkla, tydliga steg i stället för avancerade Python-genvägar, så att varje rad ska gå att förklara.

## 1. Importer

Vi använder standardbibliotek (`csv`, `json`, `datetime`) och externa bibliotek (`requests`, `matplotlib`).

In [ ]:
# --- Standardbibliotek (ingår i Python, behöver inte installeras) ---
import csv                      # för att skriva CSV-filer
import json                     # för att skriva ut JSON-data snyggt
from datetime import datetime   # för att hämta dagens datum/tid

# --- Externa bibliotek (installeras via "pip install -r requirements.txt") ---
import requests                 # för att hämta data från internet (API-anrop)
import matplotlib.pyplot as plt # för att rita diagram

print("Bibliotek importerade.")
print("Programmet startades:", datetime.now())

## 2. Utforska API:et

Innan vi bygger den färdiga lösningen testar vi API:et manuellt, för att se hur svaret ser ut. JobTech API är öppet och kräver ingen API-nyckel.

Dokumentation: https://jobsearch.api.jobtechdev.se

In [ ]:
# URL:en till JobTech APIs sökfunktion.
API_URL = "https://jobsearch.api.jobtechdev.se/search"

# "q" = vad vi söker efter. "limit" = hur många annonser vi vill ha tillbaka.
test_params = {
    "q": "AI-utvecklare",
    "limit": 5,
}

# Skickar en förfrågan till API:et och väntar på svar (max 10 sekunder).
test_response = requests.get(API_URL, params=test_params, timeout=10)

print("Statuskod:", test_response.status_code)   # 200 betyder att allt gick bra
print("Anropad URL:", test_response.url)

In [ ]:
# Kollar om anropet lyckades innan vi går vidare.
if test_response.status_code == 200:
    print("API-anropet lyckades!")
else:
    print("Något gick fel. Statuskod:", test_response.status_code)

In [ ]:
# Svaret kommer som text i JSON-format. .json() gör om det till en
# Python-dict, så att vi kan hämta ut delar av datan med [ ] eller .get().
test_data = test_response.json()

print("Typ av data:", type(test_data))
print("Nycklar i svaret:", list(test_data.keys()))

In [ ]:
# Jobbannonserna ligger i en lista under nyckeln "hits".
test_jobs = test_data.get("hits", [])
print("Antal jobbannonser i svaret:", len(test_jobs))

if len(test_jobs) > 0:
    forsta_jobbet = test_jobs[0]   # [0] = första annonsen i listan
    print("\nExempel på en jobbannons:")
    print(json.dumps(forsta_jobbet, indent=2, ensure_ascii=False)[:800], "...")
else:
    print("Inga jobbannonser hittades.")

Nu vet vi hur datan ser ut. Fälten vi vill använda är: `id`, `headline` (titel), `employer` (arbetsgivare), `workplace_address` (plats), `publication_date`, `description` (annonstexten) och `webpage_url`.

Nu bygger vi vidare med klasser och funktioner, i stället för att bara skriva ut testdata som ovan.

## 3. Klasser

En **klass** är som en mall för att skapa objekt som håller ihop information som hör samman. Vi gör en klass `JobAnnons` som representerar en jobbannons.

Sen gör vi en klass till, `AIJobbannons`, som **ärver** från `JobAnnons`. Det betyder att `AIJobbannons` automatiskt får allt som `JobAnnons` har, och vi lägger bara till det extra vi behöver (en lista med teknologier).

In [ ]:
class JobAnnons:
    # __init__ körs automatiskt varje gång vi skapar en ny JobAnnons.
    # Den sparar alla värden vi skickar in som attribut på objektet.
    def __init__(self, id, title, employer, location, published, description, url):
        self.id = id                    # annonsens id
        self.title = title              # jobbtitel, t.ex. "AI-utvecklare"
        self.employer = employer        # arbetsgivarens namn
        self.location = location        # ort/kommun
        self.published = published      # publiceringsdatum
        self.description = description  # hela annonstexten
        self.url = url                  # länk till annonsen

    def kort_beskrivning(self):
        # Denna metod bygger ihop en kort textrad om annonsen.
        return self.title + " hos " + self.employer + " (" + str(self.location) + ")"

print("Klassen JobAnnons är klar.")

Nu testar vi `JobAnnons` med lite påhittad data, bara för att se att den fungerar:

In [ ]:
test_annons = JobAnnons(
    id="123",
    title="AI-utvecklare",
    employer="Exempelföretaget AB",
    location="Stockholm",
    published="2026-01-01",
    description="Vi söker en AI-utvecklare.",
    url="https://example.com"
)

print(test_annons.kort_beskrivning())

Nu gör vi barnklassen `AIJobbannons`. Den skrivs `class AIJobbannons(JobAnnons):` — ordet inom parentesen betyder "ärver från JobAnnons".

In [ ]:
class AIJobbannons(JobAnnons):
    def __init__(self, id, title, employer, location, published, description, url, teknologier):
        # super().__init__(...) betyder "kör JobAnnons egna __init__ först".
        # Det sparar oss från att skriva samma sju rader (self.id = id, osv.) igen.
        super().__init__(id, title, employer, location, published, description, url)

        # Detta är det NYA attributet som bara AIJobbannons har.
        self.teknologier = teknologier

    def visa_teknologier(self):
        # Bygger en textsträng med alla teknologier separerade med kommatecken.
        if len(self.teknologier) == 0:
            return "Inga kända teknologier hittades."

        text = ""
        for tek in self.teknologier:
            text = text + tek + ", "
        return text

print("Klassen AIJobbannons är klar.")

Testar `AIJobbannons`. Lägg märke till att `kort_beskrivning()` fungerar även fast vi bara skrev den metoden i `JobAnnons` — det är det som menas med att **AIJobbannons ärver från JobAnnons**.

In [ ]:
test_ai_annons = AIJobbannons(
    id="456",
    title="AI-utvecklare",
    employer="Exempelföretaget AB",
    location="Stockholm",
    published="2026-01-01",
    description="Vi söker en AI-utvecklare med kunskap i Python och AWS.",
    url="https://example.com",
    teknologier=["Python", "AWS"]
)

print(test_ai_annons.kort_beskrivning())  # ärvd från JobAnnons - vi skrev den inte i AIJobbannons
print(test_ai_annons.visa_teknologier())  # finns bara i AIJobbannons

## 4. Funktioner

Vi delar upp programmet i funktioner. Varje funktion gör EN sak, har ett tydligt namn, och kan testas/förklaras för sig.

### 4.1 Teknologier vi letar efter

In [ ]:
# En vanlig lista (inte en klass eller funktion) med ord vi letar efter
# i annonstexterna. Vi skriver namnet i VERSALER för att visa att det är
# ett värde som inte ska ändras när programmet körs.
TEKNOLOGIER = ["Python", "Java", "AWS", "Azure", "Docker", "SQL", "JavaScript", "Kubernetes", "Machine Learning", "AI"]

### 4.2 Hämta data från API:et

In [ ]:
def hamta_data(sokord, antal):
    # Bygger en dict med sökparametrarna vi vill skicka till API:et.
    params = {"q": sokord, "limit": antal}

    try:
        # Försöker hämta datan från API:et.
        response = requests.get(API_URL, params=params, timeout=10)

        # Om API:et svarar med ett felmeddelande (t.ex. 404 eller 500)
        # så stoppar den här raden koden och hoppar direkt till except.
        response.raise_for_status()

    except requests.exceptions.RequestException as fel:
        # Hamnar vi här har något gått fel, t.ex. ingen internetanslutning
        # eller att API:et inte svarade i tid. Programmet kraschar INTE,
        # utan skriver bara ut vad som hände.
        print("Kunde inte hämta data för", sokord, ":", fel)
        return []   # skickar tillbaka en tom lista i stället

    # Om vi kommer hit gick allt bra. Vi plockar ut listan med annonser.
    data = response.json()
    return data.get("hits", [])

### 4.3 Leta efter teknologier i en annonstext

In [ ]:
def analysera_tech_stack(post):
    # post är en jobbannons i sitt "råa" format (en dict från API:et).
    # Vi behöver gå två steg ner för att hitta själva texten.
    beskrivning_dict = post.get("description", {})
    text = beskrivning_dict.get("text", "")

    if text is None:
        text = ""

    text_gemener = text.lower()   # gör om till små bokstäver för jämförelsen

    hittade_teknologier = []
    for tek in TEKNOLOGIER:
        if tek.lower() in text_gemener:
            hittade_teknologier.append(tek)

    return hittade_teknologier

### 4.4 Bygga AIJobbannons-objekt från rådata

In [ ]:
def skapa_jobbannons_lista(rådata):
    jobblista = []

    for post in rådata:
        teknologier = analysera_tech_stack(post)

        # Arbetsgivarens namn ligger i ett eget "under-dict".
        employer_dict = post.get("employer", {})
        foretagsnamn = employer_dict.get("name")

        # Platsen ligger också i ett eget "under-dict".
        plats_dict = post.get("workplace_address", {})
        kommun = plats_dict.get("municipality")

        beskrivning_dict = post.get("description", {})
        beskrivning_text = beskrivning_dict.get("text", "")

        # Skapar ett nytt AIJobbannons-objekt med värdena vi plockade ut.
        annons = AIJobbannons(
            id=post.get("id"),
            title=post.get("headline"),
            employer=foretagsnamn,
            location=kommun,
            published=post.get("publication_date"),
            description=beskrivning_text,
            url=post.get("webpage_url"),
            teknologier=teknologier
        )

        jobblista.append(annons)   # lägger till objektet sist i listan

    return jobblista

### 4.5 Ta bort dubbletter

In [ ]:
def ta_bort_dubbletter(rådata):
    # Vi kommer söka på flera olika sökord (se avsnitt 5), så samma
    # jobbannons kan dyka upp flera gånger. Den här funktionen ser till
    # att varje annons bara sparas en gång, genom att komma ihåg vilka
    # id:n vi redan har sett.
    sedda_id = []
    unika_annonser = []

    for post in rådata:
        post_id = post.get("id")

        if post_id not in sedda_id:
            sedda_id.append(post_id)
            unika_annonser.append(post)

    return unika_annonser

### 4.6 Spara resultat till CSV

In [ ]:
def exportera_csv(jobblista, filnamn):
    try:
        # "with open(...)" öppnar filen och stänger den automatiskt åt oss
        # när vi är klara, även om något går fel under tiden.
        with open(filnamn, mode="w", newline="", encoding="utf-8") as fil:
            skrivare = csv.writer(fil)

            # Skriver rubrikraden (kolumnnamnen) överst i filen.
            skrivare.writerow(["id", "title", "employer", "location",
                                "published", "description", "webpage_url"])

            # Skriver en rad per jobbannons.
            for annons in jobblista:
                kort_text = annons.description
                if kort_text is None:
                    kort_text = ""
                kort_text = kort_text[:300]   # bara de första 300 tecknen

                skrivare.writerow([
                    annons.id,
                    annons.title,
                    annons.employer,
                    annons.location,
                    annons.published,
                    kort_text,
                    annons.url
                ])

        print("Sparade", len(jobblista), "jobbannonser i", filnamn)

    except OSError as fel:
        # T.ex. om vi saknar rättighet att skriva filen, eller sökvägen är fel.
        print("Kunde inte spara filen:", fel)

### 4.7 Räkna teknologier och sortera resultatet

In [ ]:
def rakna_teknologier(jobblista):
    # En dict som kommer se ut ungefär som {"Python": 5, "AWS": 2, ...}
    rakning = {}

    for annons in jobblista:
        for tek in annons.teknologier:
            if tek in rakning:
                rakning[tek] = rakning[tek] + 1   # ökar räknaren med ett
            else:
                rakning[tek] = 1                  # första gången vi ser teknologin

    return rakning

def sortera_efter_antal(rakning):
    # Gör om dict:en till en lista av par: (antal, teknologi).
    # Det gör vi eftersom listor är enkla att sortera med .sort().
    lista_av_par = []
    for tek in rakning:
        antal = rakning[tek]
        lista_av_par.append((antal, tek))

    # .sort(reverse=True) sorterar listan fallande. Eftersom varje par
    # börjar med antalet sorteras listan automatiskt efter antal,
    # med flest förekomster först.
    lista_av_par.sort(reverse=True)

    return lista_av_par

### 4.8 Rita diagram

In [ ]:
def visualisera_resultat(sorterad_lista, filnamn):
    if len(sorterad_lista) == 0:
        print("Ingen data att visualisera.")
        return

    # Gör om listan av (antal, teknologi)-par till två separata listor,
    # en med namn och en med värden, som matplotlib vill ha.
    namn = []
    varden = []
    for par in sorterad_lista:
        antal = par[0]
        tek = par[1]
        namn.append(tek)
        varden.append(antal)

    plt.figure(figsize=(8, 5))              # skapar ett tomt diagram, 8x5 tum
    plt.bar(namn, varden, color="#4C72B0")  # ritar en stapel per teknologi
    plt.title("Mest efterfrågade teknologier")
    plt.xlabel("Teknologi")
    plt.ylabel("Antal förekomster")
    plt.xticks(rotation=45, ha="right")     # roterar namnen så de inte krockar
    plt.tight_layout()                      # ser till att inget hamnar utanför bilden
    plt.savefig(filnamn)                    # sparar diagrammet som en bildfil
    plt.show()                              # visar diagrammet här i notebooken

## 5. Huvudprogram

Nu kör vi alla funktioner i rätt ordning, ett steg i taget:

1. Hämta annonser för flera olika sökord
2. Ta bort dubbletter
3. Bygg AIJobbannons-objekt
4. Skriv ut några exempel
5. Spara till CSV
6. Räkna och rita diagram

In [ ]:
# Steg 1: hämta annonser för flera sökord, så vi får ett bredare urval.
sokord_lista = ["AI-utvecklare", "maskininlärning", "data scientist", "AI"]

alla_hits = []
for sokord in sokord_lista:
    traffar = hamta_data(sokord, 25)
    print("Sökord:", sokord, "->", len(traffar), "träffar")
    alla_hits = alla_hits + traffar   # slår ihop listorna till en stor lista

In [ ]:
# Steg 2: ta bort dubbletter (samma annons kan dyka upp för flera sökord).
unika_hits = ta_bort_dubbletter(alla_hits)
print("Totalt", len(alla_hits), "träffar innan dubblettborttagning")
print("Kvar efter dubblettborttagning:", len(unika_hits))

In [ ]:
# Steg 3: bygg AIJobbannons-objekt av rådatan.
jobblista = skapa_jobbannons_lista(unika_hits)

In [ ]:
# Steg 4: skriv ut några exempel för att se att det fungerar.
print("Exempel på hämtade annonser:")
for annons in jobblista[:5]:   # [:5] = bara de fem första
    print("-", annons.kort_beskrivning())

In [ ]:
# Steg 5: spara alla annonser till en CSV-fil.
exportera_csv(jobblista, "jobbdata.csv")

In [ ]:
# Steg 6: räkna hur ofta varje teknologi förekommer, sortera och rita diagram.
rakning = rakna_teknologier(jobblista)
sorterad_lista = sortera_efter_antal(rakning)

print("Teknologifrekvens (vanligast först):")
for par in sorterad_lista:
    antal = par[0]
    tek = par[1]
    print(" ", tek, ":", antal)

visualisera_resultat(sorterad_lista, "teknologifrekvens.png")